# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Queen's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

## Importing csv files

In [ ]:
import pandas as pd

demand_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv")

weather_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_holiday_block_means_with_codes.csv")

In [ ]:
#replacing Monarch's Birthday with Queen's Birthday (same day, not change in data, just title change)

demand_rank["holiday_group"] = demand_rank["holiday_group"].replace({
    "Monarch's Birthday": "Queen's Birthday"
})

weather_rank["holiday_group"] = weather_rank["holiday_group"].replace({
    "Monarch's Birthday": "Queen's Birthday"
})


In [ ]:
demand_rank.columns.tolist()

In [ ]:
#demand_rank["holiday_group"].unique()
weather_rank["holiday_group"].unique()


In [ ]:
#weather_rank.columns.tolist()
weather_rank.head(20)

# Multi-dimension plotting
- relative demand on left y axis
- two weather variables (on on x, and the other on y)
- for example, temperature on x, humidity on y

In [ ]:
def plot_dualaxis_weatherrows_for_station_4x5(
    demand_rank,
    weather_rank,
    station_code,
    holiday_name,
    right_weather_var="relative_humidity",   # second weather variable
    time_blocks=None
):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Default time blocks if not provided
    if time_blocks is None:
        time_blocks = ["00_04", "04_10", "10_15", "15_20", "20_24"]

    # Weather variables for rows (X-axis variable)
    weather_vars = [
        ("temp", "Temperature (°C)"),
        ("relative_humidity", "Relative Humidity (%)"),
        ("wind_speed_kmh", "Wind Speed (km/h)"),
        ("precip_incremental", "Precipitation (mm)")
    ]

    # Remove the row where X-axis variable equals the right-axis variable
    weather_vars = [
        (var, label) for (var, label) in weather_vars
        if var != right_weather_var
    ]


    # Filter demand
    df_d = demand_rank[
        (demand_rank["station_code"] == station_code) &
        (demand_rank["holiday_group"] == holiday_name)
    ].copy()
    if df_d.empty:
        return None, ["NO DEMAND DATA"]

    # Filter weather
    df_w = weather_rank[
        (weather_rank["holiday_group"] == holiday_name) &
        (weather_rank["station_code"] == station_code)
    ].copy()
    if df_w.empty:
        return None, ["NO WEATHER DATA"]

    # Merge
    merged = pd.merge(
        df_d,
        df_w,
        on=["date", "holiday_group", "station_code"],
        how="inner",
        suffixes=("", "_w")
    )
    if merged.empty:
        return None, ["NO MERGED DATA"]

    # Normalise flags
    merged["is_weekend"] = merged.get("is_weekend", merged.get("is_weekend_w"))
    merged["is_holiday"] = merged.get("is_holiday", merged.get("is_holiday_w"))
    merged["weekday_name"] = merged.get("weekday_name", merged.get("weekday_name_w"))

    # Station name
    full_station_name = merged["Name"].iloc[0] if "Name" in merged.columns else station_code

    # Year palette
    unique_years = sorted(merged["year"].unique())
    palette = sns.color_palette("tab20", len(unique_years))

    weekday_color = "#D8D8D8"
    weekend_color = "skyblue"

    # Create figure (NO shared Y-axis)
    fig, axes = plt.subplots(
        len(weather_vars), len(time_blocks),
        figsize=(4.5 * len(time_blocks), 4.5 * len(weather_vars)),
        sharey=False
    )

    # Column titles
    block_titles = ["12am–4am", "4am–10am", "10am–3pm", "3pm–8pm", "8pm–12am"]
    for col_idx, title in enumerate(block_titles):
        axes[0, col_idx].set_title(title, fontsize=14, pad=14)

    # Main title
    fig.suptitle(
        f"{holiday_name} — {full_station_name}\n"
        f"Demand vs Weather (Dual Axis)",
        fontsize=19,
        y=0.97,
        linespacing=0.86
    )


    missing_cols = []

    # Fill grid
    for row_idx, (xvar, x_label) in enumerate(weather_vars):
        for col_idx, block in enumerate(time_blocks):

            ax = axes[row_idx, col_idx]

            # Left Y = demand
            rank_col = f"{block}_mean_relative_rank"

            # X = weather variable for this row
            x_col = f"{block}_{xvar}"

            # Right Y = second weather variable
            right_col = f"{block}_{right_weather_var}"

            if (
                rank_col not in merged.columns or
                x_col not in merged.columns or
                merged[x_col].isna().all()
            ):
                missing_cols.append(x_col)
                ax.set_visible(False)
                continue

            # Background weekday cloud
            wd = merged[~merged["is_weekend"]]
            ax.scatter(
                wd[x_col], wd[rank_col],
                color=weekday_color, alpha=0.35, s=35
            )

            # Background weekend cloud
            we = merged[merged["is_weekend"]]
            ax.scatter(
                we[x_col], we[rank_col],
                color=weekend_color, alpha=0.45, s=35
            )

            # Holiday points (year-coloured)
            holiday_rows = merged[merged["is_holiday"]]
            sns.scatterplot(
                data=holiday_rows,
                x=x_col,
                y=rank_col,
                hue="year",
                palette=palette,
                s=110,
                ax=ax,
                legend=False,
                edgecolor="black",
                linewidth=0.6
            )

            # Left axis label only on far-left column
            if col_idx == 0:
                ax.set_ylabel("Mean Relative Rank", fontsize=12)
            else:
                ax.set_ylabel("")
                ax.tick_params(axis='y', labelleft=False)

            ax.set_xlabel(x_label, fontsize=12)
            ax.tick_params(axis='both', labelsize=10)
            ax.margins(0.05)

            # --- RIGHT AXIS (second weather variable) ---
            if right_col in merged.columns and not merged[right_col].isna().all():
                ax2 = ax.twinx()
            
                # Build a clean axis label
                if right_weather_var == "relative_humidity":
                    axis_label = "Relative Humidity (%)"
                else:
                    axis_label = right_weather_var.replace("_", " ").title()
            
                ax2.set_ylabel(axis_label, fontsize=12)
            
                # Compute limits safely
                ymin = merged[right_col].min()
                ymax = merged[right_col].max()
            
                # Prevent identical limits (flat data → singular axis)
                if ymin == ymax:
                    ymin -= 0.5
                    ymax += 0.5
            
                ax2.set_ylim(ymin, ymax)
            
                ax2.tick_params(axis='y', labelsize=10)
                ax2.margins(0.05)

    # WEEKDAY/WEEKEND LEGEND
    weekday_handle = plt.Line2D([], [], marker="o", linestyle="", color=weekday_color, markersize=8, label="Weekday")
    weekend_handle = plt.Line2D([], [], marker="o", linestyle="", color=weekend_color, markersize=8, label="Weekend")

    fig.legend(
        handles=[weekday_handle, weekend_handle],
        title="Day Type",
        title_fontsize=10.5,
        fontsize=9.5,
        loc="lower center",
        bbox_to_anchor=(0.637, 0.0001),   # <-- move to the right side
        borderpad=0.2,
        labelspacing=0.22,
        handletextpad=0.35
    )



    # YEAR LEGEND
    year_handles = [
        plt.Line2D(
            [], [], marker="o", linestyle="", markersize=8,
            markerfacecolor=palette[i], markeredgecolor="black",
            label=str(unique_years[i])
        )
        for i in range(len(unique_years))
    ]

    fig.legend(
        handles=year_handles,
        title="Year",
        title_fontsize=10.5,
        fontsize=9.5,
        loc="lower center",
        ncol=7,
        bbox_to_anchor=(0.5, 0.0001),
        borderpad=0.2,
        labelspacing=0.20,
        columnspacing=0.65,
        handletextpad=0.35
    )

    fig.subplots_adjust(
        top=0.90,
        bottom=0.08,
        left=0.06,
        right=0.98,
        hspace=0.28,
        wspace=0.32
    )

    return fig, missing_cols


In [ ]:
plot_dualaxis_weatherrows_for_station_4x5(
    demand_rank,
    weather_rank,
    "BLAKE",
    "New Year's Day",
    right_weather_var="relative_humidity"
)


## Batch function to plot all

In [ ]:
def batch_plot_all_stations_and_holidays(
    demand_rank,
    weather_rank,
    output_root="/home/565/pv3484/aus_substation_electricity/data/figures/multi_dimension_mean_rank"
):
    import os
    import matplotlib.pyplot as plt

    holidays = sorted(demand_rank["holiday_group"].dropna().unique())
    stations = sorted(demand_rank["station_code"].dropna().unique())

    os.makedirs(output_root, exist_ok=True)

    for holiday in holidays:
        print(f"\n=== {holiday} ===")

        # Create holiday-specific folder
        holiday_dir = os.path.join(output_root, holiday.replace(" ", "_"))
        os.makedirs(holiday_dir, exist_ok=True)

        for station in stations:

            outpath = os.path.join(holiday_dir, f"{station}.png")

            # --- SKIP EXISTING FILES ---
            if os.path.exists(outpath):
                print(f" → {station}: already exists, skipping")
                continue

            print(f" → Plotting {station}...")

            # Call your new multi-row, dual-axis function
            fig, missing = plot_dualaxis_weatherrows_for_station_4x5(
                demand_rank=demand_rank,
                weather_rank=weather_rank,
                station_code=station,
                holiday_name=holiday,
                right_weather_var="relative_humidity"
            )

            if fig is None:
                print(f"    Skipped — missing: {missing}")
                continue

            fig.savefig(outpath, dpi=150)
            plt.close(fig)


In [ ]:
batch_plot_all_stations_and_holidays(demand_rank, weather_rank)

# New function
- multi dimension plotting but contains R2 and pvalues per panel
- all else is same as above

In [ ]:
def plot_dualaxis_weatherrows_with_stats_4x5(
    demand_rank,
    weather_rank,
    station_code,
    holiday_name,
    right_weather_var="relative_humidity",
    time_blocks=None
):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    from scipy.stats import pearsonr

    # Default time blocks
    if time_blocks is None:
        time_blocks = ["00_04", "04_10", "10_15", "15_20", "20_24"]

    # Weather variables for rows
    weather_vars = [
        ("temp", "Temperature (°C)"),
        ("relative_humidity", "Relative Humidity (%)"),
        ("wind_speed_kmh", "Wind Speed (km/h)"),
        ("precip_incremental", "Precipitation (mm)")
    ]

    # Remove humidity row if humidity is the right-axis variable
    weather_vars = [
        (var, label) for (var, label) in weather_vars
        if var != right_weather_var
    ]

    # Filter demand
    df_d = demand_rank[
        (demand_rank["station_code"] == station_code) &
        (demand_rank["holiday_group"] == holiday_name)
    ].copy()
    if df_d.empty:
        return None, ["NO DEMAND DATA"]

    # Filter weather
    df_w = weather_rank[
        (weather_rank["holiday_group"] == holiday_name) &
        (weather_rank["station_code"] == station_code)
    ].copy()
    if df_w.empty:
        return None, ["NO WEATHER DATA"]

    # Merge
    merged = pd.merge(
        df_d,
        df_w,
        on=["date", "holiday_group", "station_code"],
        how="inner",
        suffixes=("", "_w")
    )
    if merged.empty:
        return None, ["NO MERGED DATA"]

    # Normalise flags
    merged["is_weekend"] = merged.get("is_weekend", merged.get("is_weekend_w"))
    merged["is_holiday"] = merged.get("is_holiday", merged.get("is_holiday_w"))
    merged["weekday_name"] = merged.get("weekday_name", merged.get("weekday_name_w"))

    # Station name
    full_station_name = merged["Name"].iloc[0] if "Name" in merged.columns else station_code

    # Year palette
    unique_years = sorted(merged["year"].unique())
    palette = sns.color_palette("tab20", len(unique_years))

    weekday_color = "#D8D8D8"
    weekend_color = "skyblue"

    # Create figure
    fig, axes = plt.subplots(
        len(weather_vars), len(time_blocks),
        figsize=(4.5 * len(time_blocks), 4.5 * len(weather_vars)),
        sharey=False
    )

    # Column titles
    block_titles = ["12am–4am", "4am–10am", "10am–3pm", "3pm–8pm", "8pm–12am"]
    for col_idx, title in enumerate(block_titles):
        axes[0, col_idx].set_title(title, fontsize=14, pad=14)

    # Main title
    fig.suptitle(
        f"{holiday_name} — {full_station_name}\nDemand vs Weather (Dual Axis)",
        fontsize=19,
        y=0.97,
        linespacing=0.86
    )

    missing_cols = []

    # Fill grid
    for row_idx, (xvar, x_label) in enumerate(weather_vars):
        for col_idx, block in enumerate(time_blocks):

            ax = axes[row_idx, col_idx]

            rank_col = f"{block}_mean_relative_rank"
            x_col = f"{block}_{xvar}"
            right_col = f"{block}_{right_weather_var}"

            if (
                rank_col not in merged.columns or
                x_col not in merged.columns or
                merged[x_col].isna().all()
            ):
                missing_cols.append(x_col)
                ax.set_visible(False)
                continue

            # Background weekday cloud
            wd = merged[~merged["is_weekend"]]
            ax.scatter(wd[x_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35)

            # Background weekend cloud
            we = merged[merged["is_weekend"]]
            ax.scatter(we[x_col], we[rank_col], color=weekend_color, alpha=0.45, s=35)

            # Holiday points
            holiday_rows = merged[merged["is_holiday"]]
            sns.scatterplot(
                data=holiday_rows,
                x=x_col,
                y=rank_col,
                hue="year",
                palette=palette,
                s=110,
                ax=ax,
                legend=False,
                edgecolor="black",
                linewidth=0.6
            )

            # Left axis label only on far-left column
            if col_idx == 0:
                ax.set_ylabel("Mean Relative Rank", fontsize=12)
            else:
                ax.set_ylabel("")
                ax.tick_params(axis='y', labelleft=False)

            ax.set_xlabel(x_label, fontsize=12)
            ax.tick_params(axis='both', labelsize=10)
            ax.margins(0.05)

            # --- Compute correlation statistics ---
            valid = merged[[x_col, rank_col]].dropna()
            if len(valid) > 2:
                r, p = pearsonr(valid[x_col], valid[rank_col])
                r2 = r ** 2
            else:
                r, p, r2 = np.nan, np.nan, np.nan

            # Annotate R² + p-value
            ax.text(
                0.76, 0.04,
                f"R² = {r2:.2f}\np = {p:.3f}",
                transform=ax.transAxes,
                fontsize=9,
                verticalalignment="bottom",
                bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2)
            )


            # --- RIGHT AXIS (scale only) ---
            if right_col in merged.columns and not merged[right_col].isna().all():
                ax2 = ax.twinx()

                # Label
                if right_weather_var == "relative_humidity":
                    axis_label = "Relative Humidity (%)"
                else:
                    axis_label = right_weather_var.replace("_", " ").title()

                ax2.set_ylabel(axis_label, fontsize=12)

                # Safe limits
                ymin = merged[right_col].min()
                ymax = merged[right_col].max()
                if ymin == ymax:
                    ymin -= 0.5
                    ymax += 0.5

                ax2.set_ylim(ymin, ymax)
                ax2.tick_params(axis='y', labelsize=10)
                ax2.margins(0.05)

    # WEEKDAY/WEEKEND LEGEND (bottom right)
    weekday_handle = plt.Line2D([], [], marker="o", linestyle="", color=weekday_color, markersize=8, label="Weekday")
    weekend_handle = plt.Line2D([], [], marker="o", linestyle="", color=weekend_color, markersize=8, label="Weekend")

    fig.legend(
        handles=[weekday_handle, weekend_handle],
        title="Day Type",
        title_fontsize=10.5,
        fontsize=9.5,
        loc="lower center",
        bbox_to_anchor=(0.82, 0.015),
        borderpad=0.2,
        labelspacing=0.22,
        handletextpad=0.35
    )

    # YEAR LEGEND (bottom center)
    year_handles = [
        plt.Line2D([], [], marker="o", linestyle="", markersize=8,
                   markerfacecolor=palette[i], markeredgecolor="black",
                   label=str(unique_years[i]))
        for i in range(len(unique_years))
    ]

    fig.legend(
        handles=year_handles,
        title="Year",
        title_fontsize=10.5,
        fontsize=9.5,
        loc="lower center",
        ncol=7,
        bbox_to_anchor=(0.5, 0.0001),
        borderpad=0.2,
        labelspacing=0.20,
        columnspacing=0.65,
        handletextpad=0.35
    )

    fig.subplots_adjust(
        top=0.90,
        bottom=0.08,
        left=0.06,
        right=0.98,
        hspace=0.28,
        wspace=0.38
    )

    return fig, missing_cols


In [ ]:
fig, missing = plot_dualaxis_weatherrows_with_stats_4x5(
    demand_rank,
    weather_rank,
    "BLAKE",
    "New Year's Day"
)


In [ ]:
def batch_plot_all_stations_and_holidays_r2p(
    demand_rank,
    weather_rank,
    output_root="/g/data/ng72/pv3484/substation_data/figures/multi_dimension_mean_rank/r2_pvalue"
):
    import os
    import matplotlib.pyplot as plt

    # Unique holidays and stations
    holidays = sorted(demand_rank["holiday_group"].dropna().unique())
    stations = sorted(demand_rank["station_code"].dropna().unique())

    # Ensure root exists
    os.makedirs(output_root, exist_ok=True)

    for holiday in holidays:
        print(f"\n=== {holiday} ===")

        # Folder for this holiday
        holiday_dir = os.path.join(output_root, holiday.replace(" ", "_"))
        os.makedirs(holiday_dir, exist_ok=True)

        for station in stations:

            outpath = os.path.join(holiday_dir, f"{station}.png")

            # --- SKIP EXISTING FILES ---
            if os.path.exists(outpath):
                print(f" → {station}: already exists, skipping")
                continue

            print(f" → Plotting {station}...")

            # Call your NEW stats‑enhanced function
            fig, missing = plot_dualaxis_weatherrows_with_stats_4x5(
                demand_rank=demand_rank,
                weather_rank=weather_rank,
                station_code=station,
                holiday_name=holiday,
                right_weather_var="relative_humidity"
            )

            if fig is None:
                print(f"    Skipped — missing: {missing}")
                continue

            fig.savefig(outpath, dpi=150)
            plt.close(fig)


In [ ]:
batch_plot_all_stations_and_holidays_r2p(demand_rank, weather_rank)